Demo of pipeline 
================

This notebook runs the analysis on the example data. For detailed explanations on each module, see the dedicated notebooks. 

In [ ]:
from pathlib import Path

cwd = Path.cwd()
ROOT_DIR = cwd.parent.absolute()

# Alternatively, change to your working directory:
# ROOT_DIR = Path("path/to/your/working/dir/here")

The first module is **segmentation**: it extracts clips of single organisms from images with lots of organisms. 

In [ ]:
### SEGMENTATION ###

import yaml
from mzbsuite.utils import cfg_to_arguments

arguments = {
    "input_dir": ROOT_DIR / "data/mzb_example_data/raw_img", 
    "output_dir": ROOT_DIR / "data/mzb_example_data/derived/blobs/", 
    "save_full_mask_dir": ROOT_DIR / "data/mzb_example_data/derived/full_image_masks/", 
    "config_file": ROOT_DIR / "configs/mzb_example_config.yaml", 
    "verbose": False
}

with open(str(arguments["config_file"]), "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)
cfg["trcl_gpu_ids"] = None     # not GPU-dependent

args = cfg_to_arguments(arguments)
cfg = cfg_to_arguments(cfg)

from scripts.segmentation.main_raw_to_clips import main as segmentation

segmentation(args, cfg)

100%|██████████| 2/2 [11:26<00:00, 343.24s/it]


The second module is **classification**: it uses a deep learning model to try and guess the taxonomic identity of the organism in each clip. 

In [ ]:
### CLASSIFICATION ###

MODEL_C = "convnext-small-v0"

arguments = {
    "input_dir": ROOT_DIR / "data/mzb_example_data/training_dataset/test_set/", 
    "input_model": ROOT_DIR / f"models/mzb-classification-models/{MODEL_C}",
    "taxonomy_file": ROOT_DIR / "data/mzb_example_data/MZB_taxonomy.csv",
    "output_dir": ROOT_DIR / "results/mzb_example/classification/test_set/", 
    "config_file": ROOT_DIR / "configs/mzb_example_config.yaml",
    "verbose": False
}

with open(str(arguments["config_file"]), "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)

args = cfg_to_arguments(arguments)
cfg = cfg_to_arguments(cfg)

from scripts.skeletonization.main_unsupervised_skeleton_estimation import main as skeletonization_unsupervised

skeletonization_unsupervised(args, cfg)

The **skeletonization** module comes in two variants: **unsupervised skeletonization** uses mask properties to approximate the length of organisms in clips. 

In [ ]:
### SKELETONIZATION UNSUPERVISED ###

arguments = {
    "input_dir": ROOT_DIR / "data/mzb_example_data/derived/blobs/", 
    "output_dir": ROOT_DIR / "results/mzb_example/skeletons/unsupervised_skeletons/", 
    "save_masks": ROOT_DIR / "data/mzb_example_data/derived/skeletons/unsupervised_skeletons/",
    "config_file": ROOT_DIR / "configs/mzb_example_config.yaml",
    "list_of_file": None,
    "verbose": False
}

with open(str(arguments["config_file"]), "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)
cfg["trcl_gpu_ids"] = None   # not GPU-dependent

args = cfg_to_arguments(arguments)
cfg = cfg_to_arguments(cfg)

from scripts.skeletonization.main_unsupervised_skeleton_estimation import main as skeletonization_unsupervised

skeletonization_unsupervised(args, cfg)

And the the **supervised skeletonization** module uses a deep learning model to infer organism length. 

In [ ]:
### SKELETONIZATION SUPERVISED ###

MODEL_S = "mit-b2-v0"

arguments = {
    "input_dir": ROOT_DIR / "data/mzb_example_data/derived/blobs/", 
    "input_type": "external", 
    "input_model": ROOT_DIR / f"models/mzb-classification-models/{MODEL_S}",
    "output_dir": ROOT_DIR / "results/mzb_example/skeletons/supervised_skeletons/", 
    "save_masks": ROOT_DIR / "data/mzb_example_data/derived/skeletons/supervised_skeletons/",
    "config_file": ROOT_DIR / "configs/mzb_example_config.yaml",
    "verbose": False
}

with open(str(arguments["config_file"]), "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)

args = cfg_to_arguments(arguments)
cfg = cfg_to_arguments(cfg)

from scripts.skeletonization.main_unsupervised_skeleton_estimation import main as skeletonization_unsupervised

skeletonization_unsupervised(args, cfg)

Finally, you can run a short **summarisation** script to compile all results in a CSV file.

In [ ]:
### SUMMARISATION ###

arguments = {
    "classification": ROOT_DIR / "results/mzb_example/classification/test_set/test_set_convnext-small-v0_20260303_1800",
    "skeletons_supervised": ROOT_DIR / "results/mzb_example/skeletons/supervised_skeletons/blobs_supervised_20260303_1809",
    "skeletons_unsupervised": ROOT_DIR / "results/mzb_example/skeletons/unsupervised_skeletons/blobs_unsupervised_20260303_1800", 
    "taxonomy_file": ROOT_DIR / "data/mzb_example_data/MZB_taxonomy.csv",
    "output_folder": ROOT_DIR / "results/", 
    "config_file": ROOT_DIR / "configs/mzb_example_config.yaml",
    "verbose": False
}

with open(str(arguments["config_file"]), "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)
cfg["trcl_gpu_ids"] = None

args = cfg_to_arguments(arguments)
cfg = cfg_to_arguments(cfg)

from scripts.summarisation import main as summarisation

summarisation(args, cfg)